# Solución ETL: limpieza y estandarización de datos de alumnos

En este cuaderno se desarrolla una solución completa para el proceso **ETL (Extract, Transform, Load)** del archivo `dataset_alumnos_etl.csv`.

## Objetivo
Limpiar y estandarizar las siguientes columnas:

- `rut`
- `nombre`
- `apellido_paterno`
- `apellido_materno`
- `fecha_nacimiento`
- `correo`
- `telefono`
- `carrera`

## Formato final esperado
- **rut**: `XXXXXXXX-X`
- **nombre**: primera letra mayúscula
- **apellido_paterno**: primera letra mayúscula
- **apellido_materno**: primera letra mayúscula
- **fecha_nacimiento**: `YYYY-MM-DD`
- **correo**: minúsculas, sin espacios, con `@`
- **telefono**: `+569XXXXXXXX`
- **carrera**: título por palabra, sin espacios extra


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

## 1. Extract: carga del archivo

In [2]:
ruta_entrada = "dataset_alumnos_etl.csv"
df = pd.read_csv(ruta_entrada, encoding="utf-8-sig")

print("Dimensiones del dataset:", df.shape)
df.head()

Dimensiones del dataset: (100, 8)


,rut,nombre,apellido_paterno,apellido_materno,fecha_nacimiento,correo,telefono,carrera
0,16217207-7,Camila,González,Martínez,17.02.2001,camila.gonzalez@hotmail.com,903999315,administración
1,8218062-1,FELIPE,ESPINOZA,Martínez,2007/06/25,felipe36@gmail.com,NaN,Marketing
2,17837430-3,vIcentE,Rojas,Castillo,2006-02-10,VICENTE.ROJAS@DUOC.CL,938840994,Administración
3,18.881.122-1,tOmás,Castillo,Silva,02/20/2003,tcastillo@hotmail.com,+56908883684,turismo
4,24.559.643-K,JAVIERA,Pérez,VARGAS,01/21/1999,javiera15gmail.com,980048665,MarKETiNG


## 2. Exploración inicial

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   rut               100 non-null    object
 1   nombre            100 non-null    object
 2   apellido_paterno  100 non-null    object
 3   apellido_materno  100 non-null    object
 4   fecha_nacimiento  98 non-null     object
 5   correo            93 non-null     object
 6   telefono          93 non-null     object
 7   carrera           97 non-null     object
dtypes: object(8)
memory usage: 6.4+ KB


In [4]:
df.isna().sum()

rut                 0
nombre              0
apellido_paterno    0
apellido_materno    0
fecha_nacimiento    2
correo              7
telefono            7
carrera             3
dtype: int64

In [ ]:
df.sample(10, random_state=1)

## 3. Funciones de limpieza

Se crean funciones auxiliares para limpiar y estandarizar cada columna.


In [5]:
def limpiar_texto(valor):
    if pd.isna(valor):
        return np.nan
    valor = str(valor).strip()
    if valor == "":
        return np.nan
    valor = re.sub(r"\s+", " ", valor)
    return valor

def capitalizar_texto(valor):
    valor = limpiar_texto(valor)
    if pd.isna(valor):
        return np.nan
    return valor.lower().title()

def limpiar_rut(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip().upper()
    valor = valor.replace(".", "")
    valor = valor.replace(" ", "")

    # Mantener solo números, K y guion
    valor = re.sub(r"[^0-9K\-]", "", valor)

    # Quitar guiones internos para reconstruir el formato
    valor = valor.replace("-", "")

    if len(valor) < 2:
        return np.nan

    cuerpo = valor[:-1]
    dv = valor[-1]

    if not cuerpo.isdigit():
        return np.nan

    return f"{int(cuerpo)}-{dv}"

def limpiar_fecha(valor):
    valor = limpiar_texto(valor)
    if pd.isna(valor):
        return np.nan

    # Intento 1: parseo general
    fecha = pd.to_datetime(valor, errors="coerce", dayfirst=True)

    # Intento 2: si falla, probar con monthfirst implícito
    if pd.isna(fecha):
        fecha = pd.to_datetime(valor, errors="coerce", dayfirst=False)

    if pd.isna(fecha):
        return np.nan

    return fecha.strftime("%Y-%m-%d")

def limpiar_correo(valor):
    valor = limpiar_texto(valor)
    if pd.isna(valor):
        return np.nan

    valor = valor.lower().replace(" ", "")

    # Corrección simple de doble punto
    while ".." in valor:
        valor = valor.replace("..", ".")

    # Si tiene exactamente una @ y formato básico válido, se acepta
    patron = r"^[a-z0-9._%+\-]+@[a-z0-9.\-]+\.[a-z]{2,}$"
    if re.match(patron, valor):
        return valor

    return np.nan

def limpiar_telefono(valor):
    valor = limpiar_texto(valor)
    if pd.isna(valor):
        return np.nan

    valor_mayus = str(valor).strip().upper()
    if valor_mayus in ["N/A", "NA", "NONE", "NULL", "-"]:
        return np.nan

    # Dejar solo dígitos
    digitos = re.sub(r"\D", "", str(valor))

    # Casos esperados en Chile
    # 9XXXXXXXX -> +569XXXXXXXX
    # 56 + 9XXXXXXXX -> +569XXXXXXXX
    # 569XXXXXXXX -> +569XXXXXXXX
    if len(digitos) == 9 and digitos.startswith("9"):
        return "+56" + digitos
    elif len(digitos) == 10 and digitos.startswith("09"):
        return "+56" + digitos[1:]
    elif len(digitos) == 11 and digitos.startswith("569"):
        return "+" + digitos
    elif len(digitos) == 12 and digitos.startswith("5609"):
        return "+56" + digitos[2:]
    else:
        return np.nan

def limpiar_carrera(valor):
    valor = limpiar_texto(valor)
    if pd.isna(valor):
        return np.nan
    return valor.lower().title()

## 4. Transform: limpieza y estandarización

In [6]:
df_limpio = df.copy()

# Limpiar columnas de texto
columnas_texto = ["nombre", "apellido_paterno", "apellido_materno"]
for col in columnas_texto:
    df_limpio[col] = df_limpio[col].apply(capitalizar_texto)

# Limpiar rut
df_limpio["rut"] = df_limpio["rut"].apply(limpiar_rut)

# Limpiar fecha
df_limpio["fecha_nacimiento"] = df_limpio["fecha_nacimiento"].apply(limpiar_fecha)

# Limpiar correo
df_limpio["correo"] = df_limpio["correo"].apply(limpiar_correo)

# Limpiar teléfono
df_limpio["telefono"] = df_limpio["telefono"].apply(limpiar_telefono)

# Limpiar carrera
df_limpio["carrera"] = df_limpio["carrera"].apply(limpiar_carrera)

df_limpio.head()

C:\Users\Cristian\AppData\Local\Temp\ipykernel_45012\2879786303.py:47: UserWarning: Parsing dates in %Y/%m/%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  fecha = pd.to_datetime(valor, errors="coerce", dayfirst=True)
C:\Users\Cristian\AppData\Local\Temp\ipykernel_45012\2879786303.py:47: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  fecha = pd.to_datetime(valor, errors="coerce", dayfirst=True)
C:\Users\Cristian\AppData\Local\Temp\ipykernel_45012\2879786303.py:47: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  fecha = pd.to_datetime(valor, errors="coerce", dayfirst=True)


,rut,nombre,apellido_paterno,apellido_materno,fecha_nacimiento,correo,telefono,carrera
0,16217207-7,Camila,González,Martínez,2001-02-17,camila.gonzalez@hotmail.com,+56903999315,Administración
1,8218062-1,Felipe,Espinoza,Martínez,2007-06-25,felipe36@gmail.com,NaN,Marketing
2,17837430-3,Vicente,Rojas,Castillo,2006-10-02,vicente.rojas@duoc.cl,+56938840994,Administración
3,18881122-1,Tomás,Castillo,Silva,2003-02-20,tcastillo@hotmail.com,+56908883684,Turismo
4,24559643-K,Javiera,Pérez,Vargas,1999-01-21,NaN,+56980048665,Marketing


## 5. Revisión de nulos después de la transformación

In [7]:
df_limpio.isna().sum()

rut                  0
nombre               0
apellido_paterno     0
apellido_materno     0
fecha_nacimiento     2
correo              22
telefono             7
carrera              3
dtype: int64

## 6. Eliminación de duplicados

En este paso se eliminan registros duplicados completos.  
Además, se puede revisar si existen RUT repetidos.


In [8]:
print("Filas antes de eliminar duplicados:", len(df_limpio))
df_limpio = df_limpio.drop_duplicates()
print("Filas después de eliminar duplicados:", len(df_limpio))

duplicados_rut = df_limpio["rut"].duplicated().sum()
print("Cantidad de RUT duplicados:", duplicados_rut)

Filas antes de eliminar duplicados: 100
Filas después de eliminar duplicados: 100
Cantidad de RUT duplicados: 0


## 7. Validación de formatos

In [9]:
# Patrones esperados
patron_rut = r"^\d{7,8}-[\dK]$"
patron_fecha = r"^\d{4}-\d{2}-\d{2}$"
patron_correo = r"^[a-z0-9._%+\-]+@[a-z0-9.\-]+\.[a-z]{2,}$"
patron_telefono = r"^\+569\d{8}$"

validacion = pd.DataFrame({
    "rut_valido": df_limpio["rut"].fillna("").str.match(patron_rut),
    "fecha_valida": df_limpio["fecha_nacimiento"].fillna("").str.match(patron_fecha),
    "correo_valido": df_limpio["correo"].fillna("").str.match(patron_correo),
    "telefono_valido": df_limpio["telefono"].fillna("").str.match(patron_telefono)
})

validacion.sum()

rut_valido         100
fecha_valida        98
correo_valido       78
telefono_valido     93
dtype: int64

In [10]:
df_limpio.sample(10, random_state=7)

,rut,nombre,apellido_paterno,apellido_materno,fecha_nacimiento,correo,telefono,carrera
37,22063560-0,Vicente,Torres,Rodríguez,2001-06-19,NaN,+56954776386,Ingeniería En Informática
26,23116395-6,Diego,González,Fuentes,2008-07-16,diego5@outlook.com,+56994353392,Analista Programador
78,24069917-6,Juan,Contreras,Díaz,2002-08-03,NaN,+56962579077,Turismo
91,19575184-6,Constanza,Vargas,Pérez,2008-10-29,constanza.vargas.perez@duoc.cl,+56997999171,Turismo
49,10842698-5,Vicente,Pérez,Reyes,2006-06-13,NaN,+56968248169,Ingeniería En Informática
15,25152153-0,Felipe,Torres,Contreras,2001-10-22,NaN,+56931523529,Contabilidad
93,18938238-3,Fernanda,Soto,Rodríguez,2008-04-06,fernanda.soto.rodriguez@outlook.com,+56952162278,Marketing
71,8472587-0,Felipe,Martínez,Soto,2004-08-13,felipe87@gmail.com,+56985728324,Marketing
86,20103091-9,Sofía,López,Morales,2000-01-20,sofia.lopez.morales@outlook.com,+56913721268,Analista Programador
22,24586173-7,Vicente,Morales,González,2002-01-29,vmorales@outlook.com,+56937535126,Contabilidad


## 8. Load: guardar el dataset limpio

In [11]:
ruta_salida = "dataset_alumnos_limpio.csv"
df_limpio.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

print("Archivo guardado correctamente como:", ruta_salida)

Archivo guardado correctamente como: dataset_alumnos_limpio.csv


## 9. Resumen final

Este proceso ETL realizó:

- **Extract**: carga del archivo CSV
- **Transform**:
  - limpieza de espacios
  - estandarización de mayúsculas y minúsculas
  - normalización del RUT
  - conversión de fechas a formato estándar
  - validación de correos
  - normalización de teléfonos
  - limpieza de nombres de carrera
  - eliminación de duplicados
- **Load**: exportación del dataset limpio a un nuevo archivo CSV


In [12]:
df_limpio.head(15)

,rut,nombre,apellido_paterno,apellido_materno,fecha_nacimiento,correo,telefono,carrera
0,16217207-7,Camila,González,Martínez,2001-02-17,camila.gonzalez@hotmail.com,+56903999315,Administración
1,8218062-1,Felipe,Espinoza,Martínez,2007-06-25,felipe36@gmail.com,NaN,Marketing
2,17837430-3,Vicente,Rojas,Castillo,2006-10-02,vicente.rojas@duoc.cl,+56938840994,Administración
3,18881122-1,Tomás,Castillo,Silva,2003-02-20,tcastillo@hotmail.com,+56908883684,Turismo
4,24559643-K,Javiera,Pérez,Vargas,1999-01-21,NaN,+56980048665,Marketing
5,25034970-K,Tomás,Vargas,Soto,2004-10-09,tomas82@gmail.com,+56921682744,Ingeniería En Informática
6,13540741-0,Fernanda,Pérez,Araya,2009-01-11,fernanda.perez.araya@gmail.com,+56997969689,Administración
7,25252216-6,María,Morales,Rojas,2004-03-31,maria63@gmail.com,+56977337818,Administración
8,15931579-7,Fernanda,González,Rojas,2007-02-12,fernanda62@hotmail.com,NaN,Ingeniería En Informática
9,16749844-2,Constanza,Vargas,Morales,2005-01-17,constanza.vargas.morales@gmail.com,+56961367643,Analista Programador
